# 01 — Embeddings

Generates and caches TF-IDF, MiniLM, RoBERTa (frozen), and OpenAI
embeddings of the **generated summary sentences** (not raw text) —
`00_data_transform.ipynb`'s `summary` column — for the unsupervised
clustering track. Full dataset, no sampling cap (`config.SAMPLE_SIZE`).
Downstream notebooks load from `embeddings_cache/` rather than
recomputing.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import numpy as np
import pandas as pd

from utils import config
from utils.data import stratified_sample
from utils.embeddings import (
    get_tfidf_embeddings,
    get_sentence_embeddings,
    get_bert_embeddings,
    get_openai_embeddings,
)

In [2]:
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

# config.SAMPLE_SIZE is None -> full dataset for every method (master_data
# is only ~4,750 rows total, so there's no separate smaller cap for the
# CPU-heavier RoBERTa embedding like AG News needed).
train_sample = stratified_sample(train_clean, config.SAMPLE_SIZE, seed=config.SEED)
suffix = f"n{config.SAMPLE_SIZE}" if config.SAMPLE_SIZE else "full"

print(f"Embedding {len(train_sample)} train summaries ({suffix}) and {len(test_clean)} test summaries")

Embedding 3824 train summaries (full) and 957 test summaries


In [3]:
train_tfidf = get_tfidf_embeddings(train_sample["summary"].tolist(), cache_name=f"tfidf_train_{suffix}_summary")
test_tfidf = get_tfidf_embeddings(test_clean["summary"].tolist(), cache_name="tfidf_test_full_summary")
assert train_tfidf.shape[0] == len(train_sample)
assert test_tfidf.shape[0] == len(test_clean)
print("TF-IDF dims:", train_tfidf.shape[1])

TF-IDF dims: 5000


In [4]:
train_minilm = get_sentence_embeddings(train_sample["summary"].tolist(), cache_name=f"minilm_train_{suffix}_summary")
test_minilm = get_sentence_embeddings(test_clean["summary"].tolist(), cache_name="minilm_test_full_summary")
assert train_minilm.shape[0] == len(train_sample)
print("MiniLM dims:", train_minilm.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

MiniLM dims: 384


In [5]:
train_roberta = get_bert_embeddings(train_sample["summary"].tolist(), cache_name=f"roberta_train_{suffix}_summary")
test_roberta = get_bert_embeddings(test_clean["summary"].tolist(), cache_name="roberta_test_full_summary")
assert train_roberta.shape[0] == len(train_sample)
print("RoBERTa dims:", train_roberta.shape[1])

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa dims: 768


In [6]:
if config.OPENAI_API_KEY:
    try:
        train_openai = get_openai_embeddings(train_sample["summary"].tolist(), cache_name=f"openai_train_{suffix}_summary")
        test_openai = get_openai_embeddings(test_clean["summary"].tolist(), cache_name="openai_test_full_summary")
    except Exception as e:
        print(f"OpenAI embedding failed ({type(e).__name__}): {e}")
        print("Skipping OpenAI embeddings.")
        train_openai = None
    else:
        assert train_openai.shape[0] == len(train_sample)
        print("OpenAI dims:", train_openai.shape[1])
else:
    print("OPENAI_API_KEY not set in .env — skipping OpenAI embeddings.")

OpenAI embedding failed (RateLimitError): Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
Skipping OpenAI embeddings.


In [7]:
for name, arr in [("tfidf", train_tfidf), ("minilm", train_minilm), ("roberta", train_roberta)]:
    assert not np.isnan(arr).any(), f"{name} embeddings contain NaNs"
print("No NaNs in any computed embedding matrix.")

No NaNs in any computed embedding matrix.
